**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Software-Defined Radio

> ⚠️ **Draft — code not machine-verified.** Requires an RTL-SDR USB dongle (~$30) not available at authoring time. An instructor should run each block before teaching. Remove this banner after that pass.

The most convincing demo SPS owns: a $30 USB dongle turns the entire DSP track into **real signals from real antennas** — FM stations, aircraft transponders, weather satellites. Three sessions from unboxing to decoding.

## 1. Pre-requisites

- [Digital Communications](../Intro_DSP/Digital_Communications.ipynb) and [Foundations 2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb) (multirate).
- Hardware: RTL-SDR v3/v4 dongle + telescopic antenna. Software: `pip install pyrtlsdr`, plus the `rtl-sdr` system drivers (Linux: `apt install rtl-sdr`; all platforms: see rtl-sdr.com quick-start).

---
### 🕐 Session 1 of 3 — *Hello, Spectrum* (~35 min)
**Goal:** capture live IQ samples; sweep the dial and read the local RF neighborhood.
**Feeds into:** Session 2 (FM receiver).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Hello, Spectrum</b></summary>

**Timing (~35 min).** 10 min hardware and drivers · 10 min what IQ actually is · 15 min the capture and the PSD.

**Practical warning first, and it is a serious one: this workshop needs hardware and the cells were never executed.** An RTL-SDR v3/v4 is about $30 — **buy several**, because drivers, permissions, and antenna connections all fail in ways that stall a room. On Linux, `apt install rtl-sdr` and then **blacklist the `dvb_usb_rtl28xxu` kernel module**, or the TV-tuner driver claims the device and `RtlSdr()` fails with a permissions error that looks like something else. **Test on the exact machines beforehand.**

**Then say why the logistics are worth it.** Every DSP workshop so far used synthetic signals with known answers — correct pedagogy, and also its limitation. **Here the signals are real, the noise is real, and nobody generated the ground truth.** That transition is the point, and it is the most convincing demonstration in the curriculum.

**Explain the dongle in one sentence: mixer plus sampler.** It shifts a 2.4 MHz slice of spectrum, centred wherever you tune, down to baseband and hands you **complex IQ samples**. **Everything after that is your NumPy code** — the "radio" in software-defined radio is the software.

**Make the IQ point carefully, because this is where the DSP track's groundwork gets paid.** The samples are complex not for convenience but because **a real-valued signal cannot distinguish $+f$ from $-f$**. Downconverting a band that straddles the carrier requires the analytic signal — exactly what [Foundations of Signal Processing](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) built the Hilbert transform for. **Ask why the samples are complex before answering; it is already in their notes.**

**Point at the sample rate and have the room compute the consequence.** 2.4 MS/s of complex float32 is **19 MB/s** streaming over USB. That is why the dongle drops samples when processing falls behind, and why [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb)'s budget discipline stops being hypothetical. **This capture is offline precisely so that constraint can be deferred to Session 2's exercises.**

**On the PSD cell, spend the time on `return_onesided=False`.** For complex input the spectrum is **not** symmetric, so a one-sided PSD silently discards half the band. The `fftshift` then puts DC in the middle, and adding `center_freq` converts the axis to **real megahertz**. **Three lines that turn an array index into a dial reading.**

**Have the room identify the bumps rather than showing them.** Commercial FM sits on **odd multiples of 100 kHz**, about 200 kHz wide, so 2.4 MHz spans roughly a dozen channel slots. Pull up a local station list and match peaks to call signs. **Nothing else in this curriculum lets a student point at a plot and name the thing that made it.**

**Warn about two things before the plot disappoints anyone.** `gain = "auto"` can oscillate — if the spectrum looks flat, set an explicit 30–40 dB. And the **first ~10,000 samples after tuning are junk** while the PLL settles; discard them. Both are ordinary hardware realities, and meeting them is part of the lesson.
</details>

💡 **Intuition.** The dongle is a *mixer + sampler*: it shifts a chosen slice of the RF spectrum down to baseband and hands you **complex IQ samples** — the analytic signal the DSP track has been quietly preparing you for. From here, everything is software: the 'radio' in SDR is your NumPy code.

In [ ]:
from rtlsdr import RtlSdr
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig

sdr = RtlSdr()
sdr.sample_rate = 2.4e6            # 2.4 MHz of spectrum at once
sdr.center_freq = 100.3e6          # tune near the FM band (adjust to a local station!)
sdr.gain = "auto"

iq = sdr.read_samples(256 * 1024)  # complex64 IQ
sdr.close()
print(f"captured {len(iq):,} complex samples; mean power {10*np.log10(np.mean(np.abs(iq)**2)):.1f} dB")

In [ ]:
# your RF neighborhood: Welch PSD of the capture (Statistical SP, now on airwaves)
f, P = sig.welch(iq, fs=2.4e6, nperseg=4096, return_onesided=False)
f = np.fft.fftshift(f); P = np.fft.fftshift(P)
plt.figure(figsize=(9, 3))
plt.plot((f + 100.3e6) / 1e6, 10*np.log10(P))
plt.xlabel("frequency [MHz]"); plt.ylabel("PSD [dB/Hz]")
plt.title("live spectrum: each bump is a broadcaster — count your neighbors")
plt.tight_layout(); plt.show()

**What just happened.** A Welch power spectral density of 256k live IQ samples, with the frequency axis shifted into **actual megahertz** — so every bump is a transmitter that exists, right now, near your antenna.

> ⚠️ This notebook ships **without saved outputs** (see the banner) and needs an RTL-SDR dongle. The spectrum is yours to capture.

**Three lines turn an array index into a dial reading, and each does something specific.** `return_onesided=False` is **mandatory**: the input is complex, so **the spectrum is not symmetric** — negative frequencies carry different information from positive ones, and a one-sided PSD would silently discard half the captured band. `fftshift` reorders the bins so DC sits in the middle rather than at index 0. And `+ 100.3e6` converts baseband offsets into RF frequency.

**That asymmetry is worth dwelling on, because it is the payoff of a lot of earlier theory.** A real-valued signal has a conjugate-symmetric spectrum and **cannot distinguish $+f$ from $-f$**. The dongle hands you complex samples precisely so a band straddling the tuned carrier stays unambiguous. **This is the analytic signal from [Foundations of Signal Processing](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb), arriving over USB** rather than being constructed by a Hilbert transform.

**Read the plot with a station list open, because that is the moment the workshop earns itself.** Commercial FM occupies **odd multiples of 100 kHz**, about 200 kHz wide, so 2.4 MHz spans roughly a dozen channel slots. **Match the peaks to call signs.** Nothing else in this curriculum lets a student point at a spectrum and name what produced it.

**Note why Welch rather than a plain FFT, since [Statistical Signal Processing](../Intro_DSP/Statistical_Signal_Processing.ipynb) made the argument.** A single periodogram has **variance that does not decrease with record length** — the estimate stays as noisy as the noise. Welch averages 4096-point segments, trading frequency resolution for a far smoother estimate. On real RF data that is the difference between a legible plot and a grass field.

**Sanity-check the capture before trusting the picture, using the number the previous cell printed.** A mean power sitting at the noise floor with no structure usually means **gain, not propagation** — `gain = "auto"` can oscillate, and an explicit 30–40 dB often fixes a flat-looking spectrum. And the first ~10,000 samples after tuning are junk while the PLL settles, so **discarding them before the PSD is a real improvement**.

**Two artefacts you will probably see, both of which are yours rather than the world's.** A spike at **exactly DC** — the plot's centre — is the receiver's own local-oscillator leakage, not a broadcaster, which is why practical SDR work deliberately tunes slightly *off* the signal of interest. And faint evenly-spaced spurs are USB and clock harmonics. **Distinguishing your artefacts from real signals is a skill synthetic data cannot teach.**

**Finally, note the data rate implied by that axis, because Session 2 lives inside it.** 2.4 MS/s of complex float32 is **19 MB/s** of real stream. This capture is offline, so nothing breaks; run the same processing live and every [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb) deadline applies — miss one and the dongle drops samples rather than waiting for you.

---
### 🕐 Session 2 of 3 — *A Working FM Receiver* (~40 min)
**Goal:** demodulate broadcast FM to audio in ~15 lines — every line a workshop you've taken.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (decoding digital).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: A Working FM Receiver</b></summary>

**Timing (~40 min).** 10 min the one demodulation line · 12 min the multirate plumbing · 8 min de-emphasis · 10 min listening and debugging.

**Open by promising the payoff and then delivering it in fifteen lines.** By the end of this session the room will have a `.wav` file they can play, produced entirely by their own code from radio waves their antenna caught. **That is the most memorable thing this curriculum does**, and it is worth saying up front so the multirate details feel like plumbing toward something rather than an exercise.

**Then the one line that is the whole demodulator, and derive it rather than quoting it.** FM encodes audio in the **rate of phase rotation**. So the audio is $\frac{d\phi}{dt}$, and the discrete-time version is the phase *difference* between consecutive samples. `np.angle(chan[1:] * np.conj(chan[:-1]))` computes exactly that — multiplying by the conjugate subtracts phases, and `angle` reads the result. **One line, and it is the entire radio.**

**Emphasise why the conjugate-product form is used instead of `diff(angle(chan))`.** Taking angles first gives values wrapped into $(-\pi, \pi]$, so differencing produces $2\pi$ jumps wherever the phase wraps. **The conjugate product computes the difference *before* the wrap**, so no unwrapping is needed. Ask the room to predict what the naive version sounds like; the answer is periodic clicks, and it is a genuinely instructive five-minute experiment.

**Frame everything else as multirate plumbing, and connect each step to its workshop.** 2.4 MHz → 240 kHz isolates one 200 kHz FM channel; 240 kHz → 48 kHz gets to audio rate. **`ftype="fir"` matters**: decimation without an anti-alias filter folds neighbouring stations onto your channel, which is [Foundations 2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb)'s aliasing lesson with an audible consequence. **A student who skips the filter hears the wrong station mixed into theirs.**

**Give de-emphasis its two minutes, because it is the one piece that is pure broadcast convention.** FM transmitters **boost treble** before transmission (pre-emphasis) so that high frequencies, which suffer most from noise, arrive above the floor; the receiver undoes it. The 75 µs time constant is the North American standard — **Europe uses 50 µs**, and using the wrong one gives audio that is noticeably dull or harsh. That single-pole IIR is the same filter design from [Filter Design](../Intro_DSP/Filter_Design.ipynb), with a legally specified time constant.

**Set expectations for the audio honestly, because a first capture rarely sounds perfect.** Expect it to be **mono, slightly noisy, and recognisably a radio station.** Stereo FM multiplexes a difference signal on a 38 kHz subcarrier that this receiver ignores entirely, and RDS sits at 57 kHz. **What you have built is a mono receiver, which is the right scope**, and naming the missing pieces makes them findable later.

**Have a debugging checklist ready, since something will go wrong for someone.** Silence or static usually means **wrong station** (check the Session 1 spectrum and retune to a visible peak) or **too little gain**. A harsh, distorted signal usually means **clipping** — reduce gain. Audio that is too fast or too slow means the decimation factors do not multiply to the sample rate you passed to `wavfile.write`.

**Close by counting what the room just used.** Complex baseband from [Foundations 1](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb), decimation and anti-aliasing from [Foundations 2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb), an IIR filter from [Filter Design](../Intro_DSP/Filter_Design.ipynb), and instantaneous frequency from the analytic-signal discussion. **Four workshops, fifteen lines, one working radio** — and that convergence is the argument for the whole DSP track.
</details>

💡 **Intuition.** FM encodes audio in the *rate of phase rotation* of the IQ samples. So the demodulator is one line: the angle between consecutive samples, `np.angle(iq[1:] * np.conj(iq[:-1]))`. Everything around that line is the [multirate](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb) plumbing: decimate 2.4 MHz → 240 kHz (channel), demodulate, de-emphasize, decimate → 48 kHz (audio).

In [ ]:
# the 15-line FM receiver
chan = sig.decimate(iq, 10, ftype="fir")                       # 2.4 MHz → 240 kHz
demod = np.angle(chan[1:] * np.conj(chan[:-1]))                # THE line: instantaneous frequency
# de-emphasis: FM boosts treble at the transmitter; undo it (75 µs RC in the US)
b_de, a_de = [1 - np.exp(-1/(240e3*75e-6))], [1, -np.exp(-1/(240e3*75e-6))]
audio = sig.lfilter(b_de, a_de, demod)
audio = sig.decimate(audio, 5, ftype="fir")                    # 240 kHz → 48 kHz
audio /= np.abs(audio).max()

from scipy.io import wavfile
wavfile.write("fm_capture.wav", 48000, (audio * 32767).astype(np.int16))
print("wrote fm_capture.wav — play it. that's the radio, demodulated by your own code.")

**What just happened.** A playable `.wav` file, produced from radio waves by fifteen lines of NumPy. **Play it.** That is a commercial broadcast, demodulated by code the room can read end to end.

> ⚠️ This notebook ships **without saved outputs** and needs an RTL-SDR dongle. The audio is yours to produce.

**One line is the demodulator, and it is worth deriving rather than accepting.** FM encodes audio in the **rate of phase rotation**, so the audio signal is $d\phi/dt$ — and in discrete time that is the phase *difference* between consecutive samples. `np.angle(chan[1:] * np.conj(chan[:-1]))` computes exactly that: multiplying by the conjugate **subtracts phases**, and `angle` reads the result. **The entire radio is that expression; everything else is plumbing.**

**Note why it is written that way instead of `np.diff(np.angle(chan))`, because the difference is audible.** Taking angles first wraps every value into $(-\pi, \pi]$, so differencing produces spurious $2\pi$ jumps wherever the phase crosses the branch cut. **The conjugate product takes the difference *before* the wrap**, so no unwrapping is needed. Try the naive version: it produces periodic clicks, and hearing them is a better lesson than reading this.

**The two `decimate` calls are the multirate workshop with an audible failure mode.** 2.4 MHz → 240 kHz selects a single 200 kHz FM channel; 240 kHz → 48 kHz reaches audio rate. **`ftype="fir"` is not optional**: decimating without an anti-alias filter folds neighbouring stations onto yours, which is [Foundations 2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb)'s aliasing result — **and here the symptom is literally hearing the wrong station mixed into your own.**

**De-emphasis is the one step that is pure broadcast convention rather than signal processing necessity.** FM transmitters **boost treble** before transmission, because high frequencies suffer most from channel noise; the receiver undoes the boost and attenuates the noise with it. The **75 µs** time constant is the North American standard — **Europe uses 50 µs**, and the wrong choice gives audio that is noticeably dull or harsh. Those two coefficients are a single-pole IIR from [Filter Design](../Intro_DSP/Filter_Design.ipynb), with a legally specified pole location.

**Set expectations before anyone is disappointed: this is a *mono* receiver, and correctly so.** Stereo FM multiplexes a left-minus-right signal on a **38 kHz subcarrier**, and RDS (station name, song title) sits at **57 kHz** — both are inside the 240 kHz channel and both are ignored here. **What the room built handles the mono sum**, which is the right scope for fifteen lines, and naming the missing pieces makes them findable.

**Have the debugging checklist ready, because something will go wrong for someone.** Silence or pure static: **wrong station** — go back to the Session 1 spectrum and retune to a visible peak. Harsh, distorted audio: **too much gain**, causing clipping. Audio at the wrong speed: the decimation factors ($10 \times 5 = 50$) must divide the sample rate to the 48000 passed to `wavfile.write`. **Each symptom maps to exactly one line.**

**Finally, count what this cell used, because the convergence is the argument for the whole DSP track.** Complex baseband from [Foundations 1](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb); decimation and anti-aliasing from [Foundations 2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb); an IIR filter from [Filter Design](../Intro_DSP/Filter_Design.ipynb); instantaneous frequency from the analytic-signal discussion. **Four workshops, fifteen lines, one working radio, and a $30 dongle.**

---
### 🕐 Session 3 of 3 — *Decoding Digital: ADS-B* (~40 min)
**Goal:** receive aircraft transponders at 1090 MHz; detect real packets with a matched filter.
**Builds on:** Session 2; [Statistical SP](../Intro_DSP/Statistical_Signal_Processing.ipynb) S4.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Decoding Digital — ADS-B</b></summary>

**Timing (~40 min).** 8 min the ADS-B protocol · 12 min the matched filter as detection theory · 12 min the threshold · 8 min what comes after detection.

**Open with the hardware note, because 1090 MHz is less forgiving than FM.** The stock telescopic antenna is tuned for VHF and works poorly here; **a quarter-wave whip cut to ~6.9 cm** (a paperclip works) improves results dramatically. And aircraft transmissions are **line-of-sight** — indoors near a window, or outdoors, will produce packets; a basement will not. **Manage expectations before the count comes back zero.**

**Then frame the session as detection theory arriving in the real world.** [Statistical Signal Processing](../Intro_DSP/Statistical_Signal_Processing.ipynb) Session 4 derived the matched filter as the optimal detector for a **known signal in noise**. ADS-B has a known 8 µs preamble. **This is that theorem, applied to actual aircraft**, and the room should recognise `np.correlate` as the matched filter rather than as a NumPy call.

**Walk the preamble construction, since it encodes the protocol.** At 2 MS/s each microsecond is 2 samples, so the 8 µs preamble becomes a 16-sample window with pulses at samples **0, 2, 7, 9**. **That pattern is the ADS-B standard, transcribed.** Subtracting the mean makes the template zero-mean, so the correlation responds to the *pattern* rather than to overall signal level — which matters because aircraft at different distances arrive at wildly different amplitudes.

**Note that this detector works on magnitude, not IQ, and say why that is legitimate here.** ADS-B is **pulse-position modulation** — information is in *when* energy arrives, not in its phase. Discarding phase costs nothing and removes any need for carrier synchronisation. **Contrast with the FM receiver, which was entirely phase.** Same dongle, opposite half of the complex sample, because the modulation differs.

**Make the threshold the session's methodological centrepiece, because it is Neyman–Pearson in one line.** `corr.mean() + 6 * corr.std()` sets the bar six standard deviations above the noise. **You are not choosing a detection probability; you are choosing a false-alarm rate**, and everything else follows — exactly the asymmetry the detection-theory workshop insisted on. Ask the room what happens at 3σ (many false packets, caught later by the CRC) versus 10σ (only nearby aircraft).

**Point out that the threshold is computed *from the data*, which makes it adaptive.** Because `mean` and `std` come from this capture, the bar rises automatically in a noisy environment — **that is what CFAR (constant false-alarm rate) means**, and it is why the comment says "~constant false-alarm". A fixed absolute threshold would work in one room and fail in another.

**Explain the de-duplication line, since it looks like a magic number.** A full ADS-B message is 112 bits at 1 µs per bit — **240 samples at 2 MS/s** covers the preamble plus payload. Correlation peaks are broad, so one packet produces several adjacent hits; keeping only hits separated by more than one message length collapses them. **Without it the count is inflated several-fold**, which matters because the count is the only output.

**Be honest that detection is where this notebook stops.** The cell reports *candidates*, not decoded aircraft. **Real decoding needs bit slicing, CRC validation, and field extraction** — and the CRC is what turns a candidate into a certainty, since a false alarm essentially never passes it. `pyModeS` does the field extraction; the bit slicing is a genuinely good exercise and the natural next step.

**Close with the reframing that makes the session land.** The same $30 dongle, the same NumPy, and a different template turns a spectrum browser into a **flight tracker**. **The hardware did not change; the matched filter did.** That is the clearest possible statement of what "software-defined" means, and it is worth saying explicitly as the workshop ends.
</details>

💡 **Intuition.** Aircraft broadcast position/identity as 1090 MHz pulse-position packets. The receiver is [detection theory](../Intro_DSP/Statistical_Signal_Processing.ipynb) verbatim: correlate against the known 8 µs preamble (matched filter), threshold (Neyman–Pearson), then slice bits by comparing pulse-position energies. Decoding the 56/112-bit payload (`pyModeS` does the field extraction) turns your antenna into a flight tracker.

In [ ]:
sdr = RtlSdr(); sdr.sample_rate = 2e6; sdr.center_freq = 1090e6; sdr.gain = 40
mag = np.abs(sdr.read_samples(4_000_000))          # 2 s of magnitude at 2 MS/s
sdr.close()

# ADS-B preamble at 2 MS/s: pulses at samples 0,2,7,9 in a 16-sample window
pre = np.zeros(16); pre[[0, 2, 7, 9]] = 1; pre -= pre.mean()
corr = np.correlate(mag - mag.mean(), pre, "valid")
th = corr.mean() + 6 * corr.std()                  # ~constant false-alarm threshold
hits = np.where(corr > th)[0]
# de-duplicate hits closer than one message length
msgs = hits[np.diff(hits, prepend=-999) > 240]
print(f"candidate ADS-B packets in 2 s: {len(msgs)}")
print("next step: slice bits from each hit and hand to pyModeS.decoder — see the pyModeS docs")

**What just happened.** Two seconds of 1090 MHz, correlated against a known 8 µs preamble, thresholded, de-duplicated — and the count printed is **candidate transmissions from real aircraft overhead**.

> ⚠️ This notebook ships **without saved outputs** and needs an RTL-SDR dongle. The count is yours to produce.

**This is [detection theory](../Intro_DSP/Statistical_Signal_Processing.ipynb) Session 4, applied to aircraft.** That workshop derived the **matched filter** as the optimal detector for a *known signal in noise*; ADS-B transmits a known preamble. `np.correlate(mag - mag.mean(), pre, "valid")` **is** that matched filter. The theorem and the aeroplane are the same object.

**Read the preamble construction as the protocol, transcribed.** At 2 MS/s one microsecond is two samples, so an 8 µs preamble is a 16-sample window with pulses at **0, 2, 7, 9** — the ADS-B standard, written down. Subtracting the mean makes the template zero-mean so the correlation responds to the **pattern** rather than to absolute level, which matters enormously when aircraft at 2 km and 200 km arrive at wildly different amplitudes.

**Note that this detector uses `np.abs` — magnitude only, phase discarded — and that this is correct rather than lazy.** ADS-B is **pulse-position modulation**: the information is in *when* energy arrives, not in its phase. Throwing phase away costs nothing and removes any need for carrier synchronisation. **Contrast with Session 2's FM receiver, which was entirely phase and ignored magnitude.** Same dongle, opposite half of the complex sample, because the modulation differs.

**The threshold line is Neyman–Pearson in one expression, and it deserves the emphasis.** `corr.mean() + 6*corr.std()` puts the bar six standard deviations above the noise. **You are not choosing a detection probability — you are choosing a false-alarm rate**, and sensitivity follows from it. That asymmetry is the whole content of the detection-theory workshop. Try 3σ (many false candidates) and 10σ (only nearby aircraft) and watch the count move.

**And because `mean` and `std` are computed from *this capture*, the threshold is adaptive.** In a noisy environment the bar rises automatically. **That is what CFAR — constant false-alarm rate — means**, and it is why the comment hedges with "~constant". A fixed absolute threshold would work in one room and fail in the next.

**The de-duplication line is not a magic number.** A full ADS-B message is 112 bits at 1 µs per bit, so **240 samples at 2 MS/s** spans preamble plus payload. Correlation peaks are broad, so a single packet fires several adjacent hits; `np.diff(hits, prepend=-999) > 240` keeps only hits separated by more than one message length. **Remove it and the count inflates several-fold** — which matters, since the count is the entire output.

**Be clear about what has and has not been established.** This reports **candidates**, not decoded aircraft. Some are real packets; some are noise excursions past 6σ. **The step that separates them is the CRC**, which a false alarm essentially never passes — so decoding is not just extra information, it is the *validation*. Bit slicing plus `pyModeS` is the natural next exercise and a genuinely good one.

**Two practical notes before a zero count is read as failure.** The stock telescopic antenna is tuned for VHF and is poor at 1090 MHz — **a quarter-wave whip about 6.9 cm long** (a paperclip) transforms results. And these transmissions are **line-of-sight**: near a window or outdoors gives packets, a basement does not. **Zero candidates is usually an antenna and geometry result, not a code result.**

**Finally, the reframing that closes the workshop.** Same $30 dongle, same NumPy, **different template** — and a spectrum browser becomes a flight tracker. The hardware did not change; the matched filter did. **That is what "software-defined" means**, stated as plainly as it can be.

## 4. Conclusion

IQ samples in, software radio out: the spectrum browser is Welch, the FM receiver is one `angle()` plus multirate plumbing, and the aircraft tracker is a matched filter with a CFAR-style threshold. Total hardware bill: $30.

---
## Where next

- [Digital Communications](../Intro_DSP/Digital_Communications.ipynb) — now try *transmit-side* theory against received reality.
- [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb) — make these pipelines run live instead of on captures.
- [Array Processing](../Intro_DSP/Array_Processing.ipynb) — two dongles, one shared clock: direction finding.